# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Show the dataset title and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}\nPublished: {dataset.metadata.date_published}")

## 2. Data Overview
Review available record sets, fields, and their IDs. Each record set and field is referenced by its `@id` for unambiguous access.

In [ ]:
# List all record sets by their @id
record_sets = list(dataset.metadata.record_sets)
print(f"Record sets found: {len(record_sets)}\n------------------------------")
for rs in record_sets:
    print(f"@id: {rs.id}    name: {rs.name}")
    if rs.fields:
        print("  Fields in this record set:")
        for field in rs.fields:
            print(f"    @id: {field.id}    name: {field.name}    type: {field.data_type}")
    else:
        print("  [No fields listed]")
    print("------------------------------")

## 3. Data Extraction
Load data from all available record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set using @id
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) == 0:
        print(f"No records found for record set {record_set_id}.")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for record set {record_set_id} with shape {df.shape}")

# Show the columns and first 5 records of the main record set (choose the largest for demonstration)
if len(dataframes) > 0:
    # Select the record set with the most fields or rows
    main_record_set_id = max(dataframes, key=lambda k: dataframes[k].shape[1])
    print("\nMain record set columns:", dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No tabular data available for extraction.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering, normalization, and grouping. We use field and record set `@id` for all addressing.

In [ ]:
if len(dataframes) > 0:
    df = dataframes[main_record_set_id].copy()

    # Identify numeric fields (using pandas dtypes or reasonable guess)
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if not numeric_fields:
        # Try to coerce columns that look numeric
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='ignore')
            except Exception:
                pass
        numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        threshold = df[numeric_field_id].median() if df[numeric_field_id].notnull().sum() > 0 else 10
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to choose a group field that's not numeric
        group_field_id = None
        for col in df.columns:
            if col not in numeric_fields:
                group_field_id = col
                break

        if group_field_id is not None:
            grouped_df = filtered_df.groupby(group_field_id, dropna=False)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
            print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric fields found in main record set.")
else:
    print("No main DataFrame to analyze.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. The visualization uses the main numeric field as identified in EDA.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) > 0 and numeric_fields:
    plt.figure(figsize=(8, 4))
    # Histogram of the main numeric field
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field_id} in record set {main_record_set_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id is not None:
        plt.figure(figsize=(10, 4))
        # Boxplot of main numeric field grouped by the group field
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f'{numeric_field_id} grouped by {group_field_id}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion
This notebook demonstrated loading and exploratory analysis of the FAIR^2 dataset using the `mlcroissant` library, referencing all schema entities by their `@id`. By programmatically listing and extracting all record sets and their fields, we dynamically explored the dataset content. With simple statistics and visualizations on main numeric fields, insights into their distribution and group-wise differences can guide further in-depth domain analysis. For specialized biological or proteomic questions, refer to the dataset schema and documentation to select relevant fields and mappings.